In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import scanpy as sc
import joblib

from scipy.sparse import csr_matrix

# Configuramos el estilo de las visualizaciones
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
MODELS_PATH = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'

# Nombres de los ficheros de entrada
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
REF_SC_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
MODEL_FILENAME = 'mlp_marker_genes.joblib'
OPTIMIZED_SIGNATURE_FILENAME = 'optimized_signature_matrix.parquet'

os.makedirs(os.path.join(FIGURES_PATH, 'deconvolution'), exist_ok=True)

print("Rutas definidas.")

In [ ]:
print("Cargando datos de TCGA (bulk RNA-seq)...")
bulk_counts_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
bulk_clinical_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

print("Datos de TCGA cargados:")
print(f"  - Matriz de conteos: {bulk_counts_df.shape[0]} muestras x {bulk_counts_df.shape[1]} genes")
print(f"  - Datos clínicos: {bulk_clinical_df.shape[0]} muestras x {bulk_clinical_df.shape[1]} variables")

In [ ]:
print("\nCargando datos de referencia (scRNA-seq)...")
adata_ref = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, REF_SC_FILENAME))

print("Datos de referencia cargados:")
print(adata_ref)

## 1.4. Carga de la Matriz de Firmas Optimizada
Cargamos la matriz de firmas genéticas que fue generada y optimizada en el Notebook 2. Esta matriz se basa en los 500 genes globalmente más discriminativos identificados por un modelo Random Forest.

In [ ]:
print("\nCargando la matriz de firmas optimizada...")
signature_matrix_optimized = pd.read_parquet(
    os.path.join(DATA_PROCESSED_PATH, OPTIMIZED_SIGNATURE_FILENAME)
)

print("Matriz de firmas optimizada cargada:")
print(f"Dimensiones: {signature_matrix_optimized.shape[0]} genes x {signature_matrix_optimized.shape[1]} tipos celulares")
display(signature_matrix_optimized.head())


# 2. Deconvolución de Muestras con Firma Optimizada


In [ ]:
from sklearn.svm import NuSVR
from tqdm.notebook import tqdm

In [ ]:
DECONV_RESULTS_FILENAME_FINAL = 'TCGA-LUAD_deconvolution_results_optimized.parquet'
deconv_output_path_final = os.path.join(DATA_PROCESSED_PATH, DECONV_RESULTS_FILENAME_FINAL)

if os.path.exists(deconv_output_path_final):
    print(f"--- Fichero de resultados optimizados encontrado. Cargando... ---")
    deconvolution_results_df = pd.read_parquet(deconv_output_path_final)
else:
    print(f"--- No se encontró fichero. Iniciando deconvolución con firma optimizada... ---")

In [ ]:
from sklearn.svm import NuSVR
from tqdm.notebook import tqdm

print("Extrayendo la longitud de los genes desde los metadatos del objeto de referencia...")
gene_lengths = pd.to_numeric(adata_ref.raw.var['feature_length'])

print(f"Se han obtenido las longitudes para {len(gene_lengths)} genes desde la referencia scRNA-seq.")
print(f"La matriz de bulk contiene {bulk_counts_df.shape[1]} genes.")

# Encontramos el conjunto de genes para los que tenemos toda la información
common_genes_with_length = bulk_counts_df.columns.intersection(gene_lengths.index)

print(f"Se procederá con {len(common_genes_with_length)} genes que están presentes tanto en el bulk como en la referencia con longitud conocida.")

# Filtramos la matriz de bulk ANTES de la normalización
bulk_counts_aligned = bulk_counts_df[common_genes_with_length]
# Filtramos la serie de longitudes para que coincida exactamente
gene_lengths_aligned = gene_lengths[common_genes_with_length]

In [ ]:
from scipy.sparse import csr_matrix, diags
import numpy as np

def counts_to_tpm_sparse(sparse_counts_matrix, lengths_series):
    """
    Convierte una matriz de conteos dispersa (células x genes) a TPM.
    """
    # 1. Normalizar por longitud de gen en kilobases (RPK)
    lengths_kb = lengths_series.values / 1000
    inv_lengths_kb = 1 / lengths_kb
    rpk_matrix = sparse_counts_matrix.dot(diags(inv_lengths_kb))

    # 2. Calcular el "per million" scaling factor
    # Sumamos las filas para obtener el total de RPK por célula
    sum_rpk_per_cell = np.asarray(rpk_matrix.sum(axis=1))
    
    per_million_scalers = sum_rpk_per_cell / 1e6
    
    # 3. Dividir RPK por el factor de escala para obtener TPM
    # Evitamos la división por cero
    per_million_scalers[per_million_scalers == 0] = 1
    
    # Creamos una matriz diagonal inversa para la normalización final
    inv_scalers = 1 / per_million_scalers
    # .flatten() es importante para asegurar que inv_scalers es un vector 1D
    tpm_matrix = diags(inv_scalers.flatten()).dot(rpk_matrix)
    
    return tpm_matrix.tocsr()

In [ ]:
print("\nNormalizando matriz de bulk a TPM...")
# Alinear los genes como antes
common_genes_with_length = bulk_counts_df.columns.intersection(gene_lengths.index)
bulk_counts_aligned = bulk_counts_df[common_genes_with_length]
gene_lengths_aligned = gene_lengths[common_genes_with_length]
# Convertimos a matriz dispersa de scipy
bulk_counts_sparse = csr_matrix(bulk_counts_aligned.values)
# Aplicamos la nueva función
bulk_tpm_sparse = counts_to_tpm_sparse(bulk_counts_sparse, gene_lengths_aligned)
# Creamos el DataFrame final a partir de la matriz dispersa resultante
bulk_tpm_df = pd.DataFrame.sparse.from_spmatrix(
    bulk_tpm_sparse, index=bulk_counts_aligned.index, columns=bulk_counts_aligned.columns
)

In [ ]:
# Alineamiento de Genes
print("Alineando genes entre la matriz de bulk y la nueva firma...")
# Los genes de la firma son los únicos que necesitamos
signature_genes = signature_matrix_optimized.index

In [ ]:
 # Nos aseguramos de que todos los genes de la firma están en el bulk
common_genes = list(set(signature_genes) & set(bulk_tpm_df.columns))

In [ ]:
# Filtramos ambas matrices para que coincidan
signature_matrix_aligned = signature_matrix_optimized.loc[common_genes]
bulk_tpm_aligned = bulk_tpm_df[common_genes]
    
print(f"Alineamiento completado sobre {len(common_genes)} genes.")

In [ ]:
#PRUEBA CON SUBCONJUNTO
n_samples_total = len(bulk_tpm_aligned)
#subset_size = int(n_samples_total * 0.01) # 1% de las muestras
subset_size = 10 # Aseguramos un mínimo de 3 muestras
bulk_subset_test = bulk_tpm_aligned.head(subset_size)

print(f"Subconjunto de prueba creado con {len(bulk_subset_test)} muestras.")

# 2. Ejecutar la deconvolución en el subconjunto
# (Es el mismo código del bucle principal, pero sobre el subset)
try:
    X_signature_test = signature_matrix_aligned.values
    cell_types_test = signature_matrix_aligned.columns
    all_proportions_test = []

    for sample_id in tqdm(bulk_subset_test.index, desc="Deconvolucionando subset de prueba"):
        y_bulk_sample_test = bulk_subset_test.loc[sample_id].values
        
        model_svr_test = NuSVR(kernel='linear', nu=0.5)
        model_svr_test.fit(X_signature_test, y_bulk_sample_test)
        
        raw_proportions_test = model_svr_test.coef_.copy()
        
        raw_proportions_test[raw_proportions_test < 0] = 0
        sum_proportions_test = raw_proportions_test.sum()
        if sum_proportions_test > 0:
            normalized_proportions_test = raw_proportions_test / sum_proportions_test
        else:
            normalized_proportions_test = raw_proportions_test
            
        all_proportions_test.append(normalized_proportions_test.flatten())

    # 3. Crear y verificar el DataFrame de resultados del test
    results_df_test = pd.DataFrame(
        all_proportions_test,
        index=bulk_subset_test.index,
        columns=cell_types_test
    )
    
    # Verificamos que la suma de proporciones es 1
    assert np.allclose(results_df_test.sum(axis=1), 1.0)
    
    print("\n[ÉXITO] La prueba de deconvolución se ha completado sin errores.")
    print("Las proporciones se han calculado y normalizado correctamente.")
    print("DataFrame de resultados de la prueba:")
    display(results_df_test.head())

except Exception as e:
    print(f"\n[FALLO] La prueba de deconvolución ha fallado. Error: {e}")
    # Imprimimos un traceback para ayudar a la depuración
    import traceback
    traceback.print_exc()

In [ ]:
print(results_df_test)

In [ ]:
X_signature = signature_matrix_aligned.values
cell_types = signature_matrix_aligned.columns
all_proportions = []
for sample_id in tqdm(bulk_tpm_aligned.index, desc="Deconvolucionando muestras"):
        y_bulk_sample = bulk_tpm_aligned.loc[sample_id].values
        model_svr = NuSVR(kernel='linear', nu=0.5)
        model_svr.fit(X_signature, y_bulk_sample)
        raw_proportions = model_svr.coef_.copy()
        # Post-procesamiento de los coeficientes:
        # 1. Forzar a que no sean negativos (biológicamente no tiene sentido una proporción negativa)
        raw_proportions[raw_proportions < 0] = 0
        
        # 2. Normalizar para que la suma de las proporciones sea 1 (si la suma no es cero)
        sum_proportions = raw_proportions.sum()
        if sum_proportions > 0:
            normalized_proportions = raw_proportions / sum_proportions
        else:
            normalized_proportions = raw_proportions 
            
        all_proportions.append(normalized_proportions.flatten())
deconvolution_results_df = pd.DataFrame(
        all_proportions,
        index=bulk_tpm_aligned.index,
        columns=cell_types
    )
deconvolution_results_df.columns = deconvolution_results_df.columns.astype(str)
deconvolution_results_df.to_parquet(deconv_output_path_final, engine='pyarrow')
print(f"\nResultados de la deconvolución optimizada guardados.")

In [ ]:
# Calculamos la suma de las proporciones para cada muestra
row_sums = deconvolution_results_df.sum(axis=1)

# Usamos np.allclose para verificar si todas las sumas son aproximadamente 1
try:
    assert np.allclose(row_sums, 1.0)
    print("\n[OK] Verificación exitosa: Todas las proporciones por muestra suman 1.")
except AssertionError:
    print("\n[ERROR] ¡Las proporciones no suman 1! Revisa el paso de normalización.")
    display(row_sums.describe())

In [ ]:
print("--- Fusionando resultados de deconvolución con datos clínicos ---")

# Nos aseguramos de que los índices coincidan antes de unir
try:
    assert all(deconvolution_results_df.index == bulk_clinical_df.index)
    # Usamos pd.concat para unir por columnas (axis=1)
    analysis_df = pd.concat([bulk_clinical_df, deconvolution_results_df], axis=1)
    print("Fusión completada con éxito.")
    print("Dimensiones del DataFrame de análisis final:", analysis_df.shape)
    display(analysis_df.head())
except AssertionError:
    print("[ERROR] Los índices entre los resultados de deconvolución y los datos clínicos no coinciden.")

In [ ]:
print("\n--- Visualizando la composición celular promedio ---")

# Calculamos la media de cada columna de tipo celular
mean_proportions = analysis_df[cell_types].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=mean_proportions.index, y=mean_proportions.values)
plt.title('Composición Celular Promedio en la Cohorte TCGA-LUAD', fontsize=16)
plt.ylabel('Proporción Promedio')
plt.xlabel('Tipo Celular')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(mean_proportions.to_frame(name='Proporción Promedio'))

In [ ]:
print("\n--- Comparando las proporciones de deconvolución con las de la referencia scRNA-seq ---")

# 1. Calcular las proporciones reales en el dataset de scRNA-seq de referencia
# Usamos el objeto `adata_ref` que cargamos al principio
sc_proportions = adata_ref.obs['cell_type'].value_counts(normalize=True).sort_values(ascending=False)

# 2. Obtener las proporciones promedio de la deconvolución (ya las teníamos)
deconv_proportions = mean_proportions # 'mean_proportions' de la celda anterior

# 3. Crear un DataFrame combinado para facilitar la visualización
comparison_df = pd.DataFrame({
    'Deconvolución (TCGA)': deconv_proportions,
    'Referencia (scRNA-seq)': sc_proportions
}).fillna(0) # Rellenamos con 0 si algún tipo celular no estuviera en uno de los sets

# 4. Visualizar la comparación
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Comparación de Proporciones Celulares: Deconvolución vs. Referencia scRNA-seq', fontsize=16)

# Gráfico de barras agrupado
comparison_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Composición Promedio por Método')
axes[0].set_ylabel('Proporción')
axes[0].set_xlabel('Tipo Celular')
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot para ver la correlación
sns.regplot(data=comparison_df, x='Referencia (scRNA-seq)', y='Deconvolución (TCGA)', ax=axes[1])
axes[1].set_title('Correlación entre Proporciones Estimadas y Reales')
axes[1].set_xlabel('Proporción en scRNA-seq (Real)')
axes[1].set_ylabel('Proporción en Deconvolución (Estimada)')
# Añadir línea de identidad (y=x) para una comparación perfecta
max_val = comparison_df.max().max()
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Identidad (y=x)')
axes[1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

display(comparison_df)